# 07 · Adapt OCR

Could the same invoice send a different payment amount after a scan? The synthetic invoice
**SF-177277** prints **Balance payable 1,122.29**. The lab compares the clean line with a
seeded degraded copy. A single wrong cents digit would still be a transcription error even
if almost every other character matched.

[Read the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/07-adapt-ocr)

## Which software and hardware does this lab use?

The same quantised base and processor serve every branch. Only language-backbone adapters train. The notebook stops early if CUDA is unavailable.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0",
                "bitsandbytes==0.50.2", "pillow==11.3.0", "datasets==3.6.0",
                "jiwer==4.0.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0', 'bitsandbytes==0.50.2', 'pillow==11.3.0', 'datasets==3.6.0', 'jiwer==4.0.0'], returncode=0)

## How are invoice families held out?

Original synthetic records are split by supplier-and-layout family before rendering. The same example retains its family identifier across image derivatives.

In [2]:
"""Versioned synthetic invoices and independent record scoring for the course."""

from __future__ import annotations

import json
import random
import re
from collections import Counter
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Any

CATALOG_VERSION = "course-invoice-v1"
SPLIT_SIZES = {"train": 576, "dev": 72, "familiar": 96, "novel": 96, "independent": 48, "release": 120}
VISIBLE_CONTRACT = (
    "Extract exactly one JSON object with invoice_id, issued_on (YYYY-MM-DD), supplier, "
    "currency (ISO code), line_items (description, quantity, unit_minor), subtotal_minor, "
    "tax_minor, and total_minor. Preserve line order and convert currency amounts to integer cents. "
    "Use null only for a missing or ambiguous field. Do not rewrite printed amounts, invent fields, or add explanations."
)

@dataclass(frozen=True)
class Line:
    description: str
    quantity: int
    unit_minor: int


@dataclass(frozen=True)
class Invoice:
    key: str
    family: str
    supplier: str
    issued_on: str
    currency: str
    items: tuple[Line, ...]
    tax_minor: int | None
    locale: str
    split: str
    layout: str = "register"
    printed_total_minor: int | None = None


def invoice_record(invoice: Invoice) -> dict[str, Any]:
    subtotal = sum(line.quantity * line.unit_minor for line in invoice.items)
    total = subtotal + invoice.tax_minor if invoice.tax_minor is not None else None
    if invoice.printed_total_minor is not None:
        total = invoice.printed_total_minor
    return {
        "invoice_id": invoice.key,
        "issued_on": invoice.issued_on,
        "supplier": invoice.supplier,
        "currency": invoice.currency,
        "line_items": [
            {"description": item.description, "quantity": item.quantity, "unit_minor": item.unit_minor}
            for item in invoice.items
        ],
        "subtotal_minor": subtotal,
        "tax_minor": invoice.tax_minor,
        "total_minor": total,
    }


def make_catalog() -> list[Invoice]:
    rng = random.Random(20261126)
    sources = {
        "train": (24, ("register", "memo", "checkout", "ledger")),
        "dev": (6, ("register", "memo", "checkout", "ledger")),
        "familiar": (8, ("register", "memo", "checkout", "ledger")),
        "novel": (8, ("packing", "remittance", "service", "notice")),
        "independent": (4, ("separate",)),
        "release": (10, ("packing", "remittance", "service", "notice", "separate")),
    }
    locales = (("en-US", "USD"), ("en-GB", "GBP"), ("de-DE", "EUR"), ("fr-FR", "EUR"), ("en-US", "CAD"))
    services = ("Cooling filter", "Cable sleeve", "Bench calibration", "Door seal", "Grounding clip",
                "Rack inspection", "Meter label", "Packing insert", "Carrier adjustment", "Mounting plate")
    catalog = []
    seen_keys = set()
    supplier_codes = rng.sample(range(1000, 9999), sum(n for n, _ in sources.values()))
    supplier_index = 0
    for split, (family_count, layouts) in sources.items():
        for family_number in range(family_count):
            supplier_code = supplier_codes[supplier_index]
            supplier_index += 1
            layout = layouts[family_number % len(layouts)]
            family = f"{split}-{family_number:02d}"
            locale, currency = locales[family_number % len(locales)]
            for row in range(SPLIT_SIZES[split] // family_count):
                code = f"SF-{rng.randrange(100000, 1000000)}"
                while code in seen_keys:
                    code = f"SF-{rng.randrange(100000, 1000000)}"
                seen_keys.add(code)
                items = tuple(
                    Line(kind, rng.randint(1, 9), rng.randrange(200, 24800))
                    for kind in rng.sample(services, rng.randrange(1, 4))
                )
                subtotal = sum(x.quantity * x.unit_minor for x in items)
                tax = (subtotal * (7 if currency == "USD" else 13) + 50) // 100
                if split in {"novel", "independent", "release"} and row % 12 == 4:
                    tax = None
                displayed = subtotal + tax + 101 if tax is not None and row % 12 == 7 else None
                issued_on = (date(2025, 1, 1) + timedelta(days=rng.randint(0, 590))).isoformat()
                catalog.append(Invoice(code, family, f"Sample Supplier {supplier_code}", issued_on,
                                       currency, items, tax, locale, split, layout, displayed))
    return catalog


def formatted_money(minor: int | None, locale: str) -> str:
    if minor is None:
        return "-"
    dollars, cents = divmod(minor, 100)
    number = f"{dollars:,}.{cents:02d}"
    if locale == "de-DE":
        return number.replace(",", "_").replace(".", ",").replace("_", ".")
    if locale == "fr-FR":
        return number.replace(",", " ").replace(".", ",")
    return number


def invoice_text(invoice: Invoice) -> str:
    gold = invoice_record(invoice)
    money = lambda value: formatted_money(value, invoice.locale)
    identifier = f"{invoice.key}"
    supplier = invoice.supplier
    issued = invoice.issued_on
    currency = invoice.currency
    subtotal = money(gold["subtotal_minor"])
    tax = money(gold["tax_minor"])
    total = money(gold["total_minor"])
    register = [f"{row.description} | {row.quantity} | {money(row.unit_minor)}" for row in invoice.items]
    inline = [f"{row.quantity} x {row.description} @ {money(row.unit_minor)} per unit" for row in invoice.items]
    service = [f"{row.description}: {money(row.unit_minor)} each for {row.quantity} units" for row in invoice.items]
    if invoice.layout == "register":
        return "\n".join([f"Supplier: {supplier}", f"Invoice: {identifier}", f"Issued: {issued}", f"Currency: {currency}",
                          "Description | Quantity | Unit price", *register,
                          f"Subtotal: {subtotal}", f"Tax: {tax}", f"Amount due: {total}"])
    if invoice.layout == "memo":
        return "\n".join([f"{supplier} billing memo / {issued}", f"Account invoice {identifier} [{currency}]",
                          *inline, f"Total due {total} including tax {tax}; net amount {subtotal}."])
    if invoice.layout == "checkout":
        return "\n".join([f"{identifier} / supplier {supplier}", f"Date of issue {issued} / All amounts {currency}",
                          *service, f"Net {subtotal}", f"Sales tax {tax}", f"Payable {total}"])
    if invoice.layout == "ledger":
        return "\n".join([f"{supplier} | issued {issued} | invoice {identifier}", f"Ledger in {currency}",
                          "Service | Count | Per unit", *register, f"Before tax {subtotal}",
                          f"Tax charged {tax}", f"Balance {total}"])
    if invoice.layout == "packing":
        return "\n".join([f"Billing details / {identifier}", f"Sold by {supplier}", f"Document date {issued}",
                          f"Invoice currency {currency}", f"Balance payable {total}", *service,
                          f"Merchandise {subtotal}", f"Tax {tax}"])
    if invoice.layout == "remittance":
        return "\n".join([f"Remittance advice for {identifier} ({currency})", f"{supplier} / Document issued {issued}",
                          f"Pay this amount {total}", *inline, f"Net charges {subtotal} / Tax {tax}"])
    if invoice.layout == "service":
        return "\n".join([f"Ref {identifier} / {currency}", f"Service fee notice, {issued}", supplier,
                          *[f"Unit {money(row.unit_minor)} / {row.quantity} times / {row.description}" for row in invoice.items],
                          f"Fee subtotal {subtotal}", f"Plus tax {tax}", f"Amount requested {total}"])
    if invoice.layout == "notice":
        return "\n".join([f"{supplier} / statement {identifier}", f"Issued on {issued}",
                          f"Invoice denomination {currency}", "One-off charges", *service,
                          f"Pre-tax {subtotal} / Tax assessment {tax} / Final bill {total}"])
    return "\n".join([f"Document {identifier}", f"Biller {supplier} / As of {issued} / {currency}", *inline,
                      f"Gross {total}", f"Breakdown: net {subtotal} and taxes {tax}."])


def _strict_dict(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key {key!r}")
        result[key] = value
    return result


def parse_record(completion: str | dict[str, Any]) -> dict[str, Any] | None:
    if isinstance(completion, str):
        try:
            data = json.loads(completion, object_pairs_hook=_strict_dict)
        except (ValueError, TypeError, RecursionError):
            return None
    else:
        data = completion
    keys = {"invoice_id", "issued_on", "supplier", "currency", "line_items", "subtotal_minor", "tax_minor", "total_minor"}
    if not isinstance(data, dict) or set(data) != keys:
        return None
    for name in ("invoice_id", "supplier", "issued_on", "currency"):
        value = data[name]
        if value is not None and (not isinstance(value, str) or not value.strip()):
            return None
    try:
        if data["issued_on"] is not None and date.fromisoformat(data["issued_on"]).isoformat() != data["issued_on"]:
            return None
    except ValueError:
        return None
    if data["currency"] is not None and re.fullmatch("[A-Z]{3}", data["currency"]) is None:
        return None
    for name in ("subtotal_minor", "tax_minor", "total_minor"):
        if data[name] is not None and type(data[name]) is not int:
            return None
    if not isinstance(data["line_items"], list):
        return None
    for row in data["line_items"]:
        if not isinstance(row, dict) or set(row) != {"description", "quantity", "unit_minor"}:
            return None
        if type(row["description"]) is not str or type(row["quantity"]) is not int or type(row["unit_minor"]) is not int:
            return None
        if not row["description"].strip() or row["quantity"] <= 0:
            return None
    return data


def score_record(completion: str | dict[str, Any], invoice: Invoice) -> dict[str, Any]:
    truth = invoice_record(invoice)
    observed = parse_record(completion)
    field_correct = {key: observed is not None and observed.get(key) == value
                     for key, value in truth.items() if key != "line_items"}
    keys = ("description", "quantity", "unit_minor")
    expected = Counter(tuple(r[k] for k in keys) for r in truth["line_items"])
    predicted = Counter(tuple(r[k] for k in keys) for r in observed["line_items"]) if observed is not None else Counter()
    matches = (expected & predicted).total()
    row_count = expected.total() + predicted.total()
    return {"valid": observed is not None, "exact": observed == truth,
            "fields": sum(field_correct.values()) / len(field_correct), "field_correct": field_correct,
            "row_f1": 2 * matches / row_count if row_count else 1.0}


def rules_extract(text: str) -> dict[str, Any] | None:
    lines = text.splitlines()
    if len(lines) < 9 or lines[4] != "Description | Quantity | Unit price":
        return None
    property_names = ("supplier", "invoice_id", "issued_on", "currency")
    headers = ("Supplier: ", "Invoice: ", "Issued: ", "Currency: ")
    if any(not line.startswith(header) for line, header in zip(lines, headers)):
        return None
    data = {name: line.removeprefix(header) for name, line, header in zip(property_names, lines, headers)}

    def minor(text_amount: str) -> int | None:
        if text_amount == "-":
            return None
        if re.fullmatch(r"(?:\d+|\d{1,3}(?:[., ]\d{3})+)[.,]\d{2}", text_amount) is None:
            raise ValueError("Unsupported amount format")
        units, frac = text_amount[:-3], text_amount[-2:]
        return int(re.sub(r"\D", "", units)) * 100 + int(frac)

    try:
        rows = [line.split(" | ") for line in lines[5:-3]]
        data["line_items"] = [{"description": desc, "quantity": int(qty), "unit_minor": minor(unit)}
                              for desc, qty, unit in rows]
        data["subtotal_minor"] = minor(lines[-3].removeprefix("Subtotal: "))
        data["tax_minor"] = minor(lines[-2].removeprefix("Tax: "))
        data["total_minor"] = minor(lines[-1].removeprefix("Amount due: "))
    except ValueError:
        return None
    return parse_record(data)


def examples_for(split: str, limit: int | None = None) -> list[Invoice]:
    found = [invoice for invoice in make_catalog() if invoice.split == split]
    if not found:
        raise ValueError(f"Unknown invoice split {split!r}")
    if limit is None or limit >= len(found):
        return found
    if limit <= 0:
        raise ValueError("Limit must be positive")
    groups: dict[str, list[Invoice]] = {}
    for invoice in found:
        groups.setdefault(invoice.family, []).append(invoice)
    balanced = [invoice for group_rows in zip(*groups.values(), strict=True) for invoice in group_rows]
    return balanced[:limit]

In [3]:
"""Original invoice page and chart rendering for the document notebooks."""

from PIL import Image, ImageDraw, ImageFont

# The notebooks inline this file after the fixture contract. The fallback import
# keeps the same functions usable in ordinary Python.
if "invoice_text" not in globals():
    from invoice_fixtures import Invoice, invoice_record, invoice_text


def _font(size: int, bold: bool = False):
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    try:
        return ImageFont.truetype(name, size)
    except OSError:
        return ImageFont.load_default(size=size)


def invoice_image(invoice: Invoice) -> Image.Image:
    image = Image.new("RGB", (896, 624), "#faf9f5")
    draw = ImageDraw.Draw(image)
    fields = invoice_text(invoice).splitlines()
    accents = {
        "register": "#245067", "memo": "#732f41", "checkout": "#30614e", "ledger": "#2a3e72",
        "packing": "#31535c", "remittance": "#694323", "service": "#4b3f65", "notice": "#73422d",
    }
    accent = accents.get(invoice.layout, "#4a5744")
    draw.rounded_rectangle((32, 29, 864, 595), 11, fill="white", outline="#d9dad8", width=2)
    draw.rectangle((33, 34, 863, 42), fill=accent)
    draw.text((64, 63), "Supplier billing", font=_font(28, bold=True), fill=accent)
    draw.text((64, 102), "Original synthetic course fixture", font=_font(15), fill="#5b6462")
    draw.line((61, 134, 835, 134), fill="#d5dad8", width=2)
    for i, line in enumerate(fields):
        y = 153 + 35 * i
        if i % 2:
            draw.rectangle((58, y - 3, 837, y + 28), fill="#f5f7f6")
        is_heading = line.startswith(("Description |", "Service |", "One-off charges"))
        draw.text((65, y), line, fill="#233034", font=_font(20, bold=is_heading))
        if draw.textbbox((65, y), line, font=_font(20, bold=is_heading))[2] >= 837:
            raise ValueError(f"Invoice field did not fit the page: {line!r}")
    return image


def spend_chart(invoice: Invoice, *, varied: bool = False):
    """Render line-item spending as labelled bars with separately known cents."""
    gold = invoice_record(invoice)
    values = [row["quantity"] * row["unit_minor"] for row in gold["line_items"]]
    image = Image.new("RGB", (896, 512), "#ffffff")
    draw = ImageDraw.Draw(image)
    accent = "#266978" if varied else "#27598b"
    order = list(reversed(range(len(values)))) if varied else list(range(len(values)))
    draw.text((48, 39), f"Line item spending · {invoice.currency}", fill=accent, font=_font(28, bold=True))
    draw.text((48, 80), "Total before tax per line item", fill="#53575b", font=_font(17))
    largest = max(values)
    rows = []
    for position, index in enumerate(order):
        row = gold["line_items"][index]
        value = values[index]
        y = 141 + 86 * position
        draw.text((51, y), row["description"], font=_font(19), fill="#222222")
        money = f"{value / 100:,.2f}"
        draw.text((734, y), money, font=_font(19, bold=True), fill="#222222")
        draw.rounded_rectangle((51, y + 34, 51 + int(value / largest * 650), y + 55), 4, fill=accent)
        rows.append({"description": row["description"], "line_total_minor": value})
    return image, {"currency": invoice.currency, "rows": rows}

## Which tokens receive a loss?

Both renders use the same image tokens. The prompt prefix must match the full rendered sequence exactly before the mask removes it.

In [4]:
"""Bounded Qwen2-VL training and evaluation shared by the document notebooks."""

from dataclasses import dataclass
from pathlib import Path
import random
from time import perf_counter

import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"
VISION_REV = "1860c46f12b7449d0f5b426a7474ab715342bc65"
VISION_SEED = 42
VISION_MAX_PIXELS = 576 * 28 * 28
VISION_MAX_OUTPUT = 220


@dataclass
class DocumentExample:
    key: str
    image: object
    instruction: str
    answer: str
    group: str


def processor_for_vision():
    return AutoProcessor.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False,
        min_pixels=4 * 28 * 28, max_pixels=VISION_MAX_PIXELS,
    )


def load_vision_base(trainable: bool = False):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime. This vision lab uses CUDA.")
    quantisation = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False, use_safetensors=True,
        torch_dtype=torch.float16, device_map={"": 0}, quantization_config=quantisation,
        attn_implementation="sdpa",
    )
    if trainable:
        return prepare_model_for_kbit_training(base, gradient_checkpointing_kwargs={"use_reentrant": False})
    return base.eval()


def adapter_for_vision(base):
    adapter = get_peft_model(
        base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                         lora_dropout=0.0, task_type="CAUSAL_LM"),
    )
    trainable = [name for name, value in adapter.named_parameters() if value.requires_grad]
    if not trainable or any(".visual." in name or "lora" not in name for name in trainable):
        raise RuntimeError("Expected language-backbone LoRA parameters only.")
    return adapter


def encode_image(processor, item: DocumentExample, assistant: str | None = None):
    chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": item.instruction}]}]
    if assistant is not None:
        chat.append({"role": "assistant", "content": [{"type": "text", "text": assistant}]})
    template = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=assistant is None)
    # Use the exact same pixels and processor settings in the prompt and full sequence.
    inputs = processor(text=[template], images=[item.image], padding=False, return_tensors="pt")
    return inputs.to("cuda")


def sft_feature(processor, item: DocumentExample):
    prompt = encode_image(processor, item)
    full = encode_image(processor, item, item.answer)
    labels = full["input_ids"].clone()
    prefix = prompt["input_ids"]
    width = prefix.shape[1]
    if width >= labels.shape[1] or not torch.equal(labels[:, :width], prefix):
        raise ValueError(f"Chat template changed its expanded image prefix for {item.key}.")
    labels[:, :width] = -100
    if not torch.any(labels[:, width:] != -100):
        raise ValueError("The prompt mask removed the entire target.")
    return full, labels


def fit_vision(adapter, processor, items, *, updates: int, accumulation: int = 2):
    random.seed(VISION_SEED)
    order = list(items)
    random.shuffle(order)
    if updates <= 0 or not order:
        raise ValueError("Vision training needs a positive update budget and at least one example.")
    adapter.train()
    optimiser = torch.optim.AdamW((p for p in adapter.parameters() if p.requires_grad), lr=1e-4)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = perf_counter()
    history = []
    for update in range(updates):
        optimiser.zero_grad(set_to_none=True)
        step_loss = 0.0
        for j in range(accumulation):
            features, labels = sft_feature(processor, order[(update * accumulation + j) % len(order)])
            scalar = adapter(**features, labels=labels, use_cache=False).loss
            if not torch.isfinite(scalar):
                raise RuntimeError(f"Non-finite vision SFT loss at step {update}.")
            (scalar / accumulation).backward()
            step_loss += scalar.detach().item() / accumulation
        norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        if not torch.isfinite(norm):
            raise RuntimeError("Non-finite adapter gradient.")
        optimiser.step()
        history.append(step_loss)
    torch.cuda.synchronize()
    adapter.eval()
    return {"updates": updates, "examples_seen": updates * accumulation, "seconds": perf_counter() - started,
            "peak_gib": torch.cuda.max_memory_allocated() / 2**30, "loss": history}


@torch.inference_mode()
def generate_vision(model, processor, example, *, max_new_tokens=VISION_MAX_OUTPUT, **options):
    inputs = encode_image(processor, example)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, **options)
    ids = output[0, inputs["input_ids"].shape[1]:]
    value = processor.tokenizer.decode(ids, skip_special_tokens=True).strip()
    stops = model.generation_config.eos_token_id
    eos = stops if isinstance(stops, (list, tuple)) else [stops]
    return {"text": value, "tokens": ids.cpu().tolist(), "complete": bool(len(ids) and ids[-1].item() in eos)}


def save_vision_adapter(model, path: str) -> Path:
    target = Path(path)
    model.save_pretrained(target, safe_serialization=True)
    if not (target / "adapter_model.safetensors").is_file():
        raise RuntimeError(f"Missing adapter weights at {target}")
    return target

In [5]:
import json
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
import hashlib
import random
import shutil

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook needs a Colab T4 runtime.")
torch.manual_seed(VISION_SEED)
START = perf_counter()
print("GPU", torch.cuda.get_device_name(0))
for package in ("torch", "transformers", "peft", "bitsandbytes", "Pillow"):
    print(package, version(package))

def paired_family_exact(before, after, get_exact, seed=42):
    baseline = {row["key"]: row for row in before}
    assert len(baseline) == len(before) == len(after)
    assert set(baseline) == {row["key"] for row in after}
    deltas = {}
    for current in after:
        previous = baseline[current["key"]]
        assert previous["family"] == current["family"]
        deltas.setdefault(current["family"], []).append(get_exact(current) - get_exact(previous))
    families = sorted(deltas)
    if len(families) < 2:
        return {"families":len(families),"interval_pp":None}
    rng = random.Random(seed)
    draws = []
    for _ in range(2000):
        rows = [delta for family in rng.choices(families, k=len(families)) for delta in deltas[family]]
        draws.append(100 * sum(rows) / len(rows))
    draws.sort()
    return {"families":len(families),
            "delta_pp":100 * sum(sum(values) for values in deltas.values()) / len(after),
            "interval_pp":[draws[49],draws[1949]]}

def feature_manifest(items, updates, accumulation):
    return {
        "model":VISION_MODEL, "revision":VISION_REV, "catalogue":CATALOG_VERSION,
        "updates":updates, "accumulation":accumulation, "seed":VISION_SEED,
        "features_sha256":hashlib.sha256(json.dumps([
            [item.key, item.group, item.instruction, item.answer,
             hashlib.sha256(item.image.tobytes()).hexdigest()] for item in items
        ],ensure_ascii=False).encode()).hexdigest(),
    }

def write_results(filename, payload):
    model = {"id": VISION_MODEL, "revision": VISION_REV}
    versions = {p: version(p) for p in ("torch","transformers","peft","bitsandbytes","Pillow","datasets")}
    experiment = {"elapsed_seconds": round(perf_counter()-START, 3),
                  "gpu": torch.cuda.get_device_name(0), "model": model,
                  "packages": versions, "catalogue": CATALOG_VERSION, "seed": VISION_SEED}
    with open(filename, "w", encoding="utf-8") as handle:
        json.dump({**experiment, **payload}, handle, indent=2, ensure_ascii=False)
    print(json.dumps({**experiment, **payload}, ensure_ascii=False))

GPU Tesla T4
torch 2.14.1+cu130
transformers 4.51.3
peft 0.15.2
bitsandbytes 0.50.2
Pillow 11.3.0


## How do we vary the input?

Keep the document and label fixed. The varied branch receives a small rotation, blur and
contrast reduction. Compare pixels from the two exports for the opening invoice before
inspecting predictions.

In [6]:
"""OCR fixtures, row-level error accounting and the bounded T4 profile."""

from collections import defaultdict
from gc import collect
from PIL import ImageEnhance, ImageFilter, ImageOps
from jiwer import process_characters, process_words
import subprocess
import tempfile

OCR_PROMPT = "Transcribe the single line in this image exactly. Output the line only."
CORD_REV = "7f0115a4b758a71d6473b8d085751692da2fef98"
OPENING_KEY = "SF-177277"


def degrade(image, key):
    rng = random.Random(key)
    return ImageEnhance.Contrast(
        image.rotate(rng.choice((-1.5, -0.8, 0.8, 1.5)), fillcolor="white")
             .filter(ImageFilter.GaussianBlur(rng.uniform(0.5, 1.2)))
    ).enhance(rng.uniform(0.58, 0.85))


def matched_crops(split, limit):
    clean, varied = [], []
    for invoice in examples_for(split, limit):
        lines = invoice_text(invoice).splitlines()
        candidates = [i for i, line in enumerate(lines)
                      if not line.startswith(("Description |", "Service |", "One-off charges"))]
        index = next(i for i, line in enumerate(lines) if line.startswith("Balance payable")) if invoice.key == OPENING_KEY else random.Random(invoice.key).choice(candidates)
        image = invoice_image(invoice)
        top = 150 + 35 * index
        assert top + 34 <= image.height, f"Crop is outside {invoice.key}"
        crop = image.crop((58, top, 838, top + 34))
        varied_crop = degrade(crop, invoice.key)
        clean.append(DocumentExample(invoice.key, crop, OCR_PROMPT, lines[index], invoice.family))
        varied.append(DocumentExample(invoice.key, varied_crop, OCR_PROMPT, lines[index], invoice.family))
    keys = lambda rows: [(item.key, item.group, item.answer) for item in rows]
    assert keys(clean) == keys(varied) and len(set(keys(clean))) == limit
    return clean, varied


def cord_crop(raw, number):
    ground = json.loads(raw["ground_truth"])
    image = raw["image"].convert("RGB")
    for line in ground["valid_line"]:
        words = [w for w in line["words"] if w.get("text", "").strip() and w.get("quad")]
        if len(words) < 2:
            continue
        quads = [word["quad"] for word in words]
        left = max(0, int(min(q[f"x{i}"] for q in quads for i in range(1, 5))) - 4)
        right = min(image.width, int(max(q[f"x{i}"] for q in quads for i in range(1, 5))) + 5)
        top = max(0, int(min(q[f"y{i}"] for q in quads for i in range(1, 5))) - 3)
        bottom = min(image.height, int(max(q[f"y{i}"] for q in quads for i in range(1, 5))) + 4)
        if right - left > 10 and bottom - top > 5:
            key = f"cord-{number}"
            crop = image.crop((left, top, right, bottom))
            # The pinned Qwen processor requires both dimensions to span one merged patch.
            horizontal = max(0, 28 - crop.width)
            vertical = max(0, 28 - crop.height)
            crop = ImageOps.expand(crop, (horizontal // 2, vertical // 2, (horizontal + 1) // 2,
                                         (vertical + 1) // 2), fill="white")
            return DocumentExample(key, crop, OCR_PROMPT,
                                   " ".join(w["text"] for w in words), key)
    return None


def tesseract_text(item):
    with tempfile.TemporaryDirectory() as location:
        path = Path(location) / "crop.png"
        item.image.save(path)
        return {"text": subprocess.check_output(
            ["tesseract", str(path), "stdout", "--psm", "7", "-l", "ind+eng"],
            stderr=subprocess.DEVNULL, text=True,
        ).strip(), "complete": None}


def error_counts(process, truth, prediction):
    counts = process(truth, prediction)
    edits = counts.substitutions + counts.deletions + counts.insertions
    reference_length = counts.hits + counts.substitutions + counts.deletions
    return edits, reference_length


def measure_ocr(test_sets, method):
    answer = {}
    for split, items in test_sets.items():
        rows = []
        for item in items:
            raw = method(item)
            truth = " ".join(item.answer.split())
            prediction = " ".join(raw["text"].split())
            char_edits, chars = error_counts(process_characters, truth, prediction)
            word_edits, words = error_counts(process_words, truth, prediction)
            rows.append({"key": item.key, "family": item.group, "truth": truth, "prediction": prediction,
                         "raw_truth": item.answer, "raw_prediction": raw["text"], "complete": raw["complete"],
                         "char_edits": char_edits, "chars": chars, "word_edits": word_edits, "words": words,
                         "exact": truth == prediction})
        answer[split] = {
            "count": len(rows), "exact": sum(row["exact"] for row in rows) / len(rows),
            "cer": sum(row["char_edits"] for row in rows) / sum(row["chars"] for row in rows),
            "wer": sum(row["word_edits"] for row in rows) / sum(row["words"] for row in rows), "rows": rows,
        }
    return answer


def paired_error_changes(before, after):
    assert set(before) == set(after)
    changes = {}
    for split, baseline in before.items():
        pairs = defaultdict(list)
        for a, b in zip(baseline["rows"], after[split]["rows"], strict=True):
            assert (a["key"], a["family"], a["truth"]) == (b["key"], b["family"], b["truth"])
            pairs[a["family"]].append((a, b))
        families = sorted(pairs)
        results = {}
        for metric, edits, count in (("cer", "char_edits", "chars"), ("wer", "word_edits", "words")):
            totals = [(sum(b[edits] - a[edits] for a, b in pairs[key]),
                       sum(a[count] for a, _ in pairs[key])) for key in families]
            rng = random.Random(42)
            draws = []
            for _ in range(2000):
                sampled = rng.choices(totals, k=len(totals))
                draws.append(100 * sum(delta for delta, _ in sampled) / sum(n for _, n in sampled))
            draws.sort()
            results[metric] = {"delta_pp": 100 * (after[split][metric] - baseline[metric]),
                               "interval_pp": [draws[49], draws[1949]] if len(families) >= 2 else None}
        changes[split] = {"families": len(families), "results": results}
    return changes


def profile_t4(processor, candidates):
    free, _ = torch.cuda.mem_get_info()
    baseline = torch.cuda.memory_allocated()
    if free < 12 * 2**30:
        raise RuntimeError("OCR profile requires at least 12 GiB free on T4.")
    torch.manual_seed(VISION_SEED)
    adapter = adapter_for_vision(load_vision_base(trainable=True))
    # Process on CPU to measure sequence lengths without growing CUDA memory.
    def width(item):
        chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": item.instruction}]},
                {"role": "assistant", "content": [{"type": "text", "text": item.answer}]}]
        template = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
        return processor(text=[template], images=[item.image], return_tensors="pt")["input_ids"].shape[1]
    largest = max(candidates, key=width)
    profile = fit_vision(adapter, processor, [largest], updates=1, accumulation=2)
    profile["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 2**30
    profile["longest_prompt_and_response_tokens"] = width(largest)
    profile["key"] = largest.key
    del adapter
    collect()
    torch.cuda.empty_cache()
    torch.manual_seed(VISION_SEED)
    if (profile["seconds"] > 45 or profile["peak_gib"] > 10
        or profile["peak_reserved_gib"] > min(12, free / 2**30 - 2)
        or torch.cuda.memory_allocated() > baseline + 128 * 2**20):
        raise RuntimeError(f"OCR training profile exceeds the T4 budget: {profile}")
    return profile

In [7]:
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
clean_train, varied_train = matched_crops("train", 72)
clean_test, varied_test = matched_crops("novel", 32)
assert {x.group for x in clean_train}.isdisjoint(x.group for x in clean_test)
assert {x.key for x in clean_train}.isdisjoint(x.key for x in clean_test)
opening = [next(x for x in items if x.key == OPENING_KEY) for items in (clean_test, varied_test)]
assert opening[0].answer == "Balance payable 1,122.29"
for name, item in zip(("clean", "degraded"), opening):
    item.image.save(RESULTS / f"07-opening-{name}.png")
print("Opening source line:", opening[0].answer)
print("Matched training and novel counts:", len(clean_train), len(clean_test))

Opening source line: Balance payable 1,122.29
Matched training and novel counts: 72 32


## How is the receipt transfer slice built?

Take one annotated text line from each selected receipt in the official CORD v2 test set.
The crop covers the annotation's word coordinates, and the target follows the annotated line
order. Pad crops below 28 pixels on either side with white so the pinned Qwen processor accepts
them. Every method receives the same padded pixels. [CORD dataset and licence](https://huggingface.co/datasets/naver-clova-ix/cord-v2)

In [8]:
from datasets import load_dataset

stream = load_dataset("naver-clova-ix/cord-v2", revision=CORD_REV, split="test", streaming=True)
external = []
for i, row in enumerate(stream.take(48)):
    item = cord_crop(row, i)
    if item is not None:
        external.append(item)
        if len(external) == 20:
            break
assert len(external) == 20, "The pinned receipt selection changed."
external[0].image.save(RESULTS / "07-transfer-example.png")
test_sets = {"clean_novel": clean_test, "degraded_novel": varied_test, "cord_lines": external}
print("Receipt transfer crops:", len(external))

Receipt transfer crops: 20


## Can a fixed OCR engine read these same lines?

Tesseract uses page segmentation mode 7 for one line. Use the same Indonesian and English
language configuration across all three slices. Whitespace normalisation is shared across
all systems. Case, punctuation and cents remain in the scoring inputs.

In [9]:
import subprocess

def installed_languages():
    if not shutil.which("tesseract"):
        return set()
    return set(subprocess.check_output(["tesseract", "--list-langs"], text=True).splitlines()[1:])

if not {"eng", "ind"} <= installed_languages():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr", "tesseract-ocr-eng", "tesseract-ocr-ind"], check=True)
assert {"eng", "ind"} <= installed_languages()
ocr_version = subprocess.check_output(["tesseract", "--version"], text=True).splitlines()[0]
print(ocr_version, "languages: ind+eng; psm: 7")
scores = {"tesseract": measure_ocr(test_sets, tesseract_text)}

tesseract 5.3.4 languages: ind+eng; psm: 7


## Can the T4 fit the largest prepared crop?

Run one training example selected by the measured expanded input length across both
training variants. Fail early on a memory or time budget breach. Both scored arms then
reload the same pinned base with a fresh optimiser and seed.

In [10]:
from gc import collect

processor = processor_for_vision()
profile = profile_t4(processor, clean_train + varied_train)
(RESULTS / "07-t4-profile.json").write_text(json.dumps(profile, indent=2) + "\n")
print("T4 profile:", json.dumps(profile))

T4 profile: {"updates": 1, "examples_seen": 2, "seconds": 3.922286530999827, "peak_gib": 2.0678606033325195, "loss": [2.921293020248413], "peak_reserved_gib": 3.248046875, "longest_prompt_and_response_tokens": 99, "key": "SF-540371"}


## What does the frozen checkpoint read?

Use the identical generation cap and greedy decoding for the frozen vision model and both
adapters.

In [11]:
def scored_transcript(model, item):
    return generate_vision(model, processor, item, max_new_tokens=64)

base = load_vision_base()
scores["frozen"] = measure_ocr(test_sets, lambda item: scored_transcript(base, item))
del base
collect()
torch.cuda.empty_cache()

## Does the same update budget help degraded crops?

Both branches see the same invoice IDs and transcripts in the same order. The evaluated
treatment is the training pixels. Keep the clean and degraded test scores separate.

In [12]:
runs = {}
for label, items in (("clean_adapter", clean_train), ("varied_adapter", varied_train)):
    torch.manual_seed(VISION_SEED)
    model = adapter_for_vision(load_vision_base(trainable=True))
    runs[label] = fit_vision(model, processor, items, updates=24, accumulation=2)
    scores[label] = measure_ocr(test_sets, lambda item: scored_transcript(model, item))
    del model
    collect()
    torch.cuda.empty_cache()

## Read the result

Inspect the learner's opening crop, then compare paired changes with supplier families (or
receipt IDs) as the resampling unit. Negative CER/WER changes mean fewer errors.
[Read the lesson after measuring](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/07-adapt-ocr)

In [13]:
intervals = paired_error_changes(scores["clean_adapter"], scores["varied_adapter"])
opening_rows = {
    name: {split: next(row for row in result["rows"] if row["key"] == OPENING_KEY)
           for split, result in by_slice.items() if split != "cord_lines"}
    for name, by_slice in scores.items()
}
features, labels = sft_feature(processor, opening[0])
mask = labels[0] != -100
tokeniser = processor.tokenizer
visual_tokens = tokeniser.convert_tokens_to_ids("<|image_pad|>")
mask_evidence = {
    "image_tokens": int((features["input_ids"][0] == visual_tokens).sum()),
    "masked_prefix_tokens": int((~mask).sum()),
    "first_supervised_token": tokeniser.decode(labels[0][mask][:1]),
}
print(json.dumps(opening_rows, indent=2))
print("Opening crop mask:", json.dumps(mask_evidence))
write_results(str(RESULTS / "07-ocr-results.json"), {
    "scores": scores, "training": runs, "paired_changes": intervals, "profile": profile,
    "cord_revision": CORD_REV, "tesseract_version": ocr_version, "tesseract_languages": "ind+eng",
    "tesseract_psm": 7, "max_new_tokens": 64, "opening": opening_rows,
    "opening_mask": mask_evidence, "transfer_example_key": external[0].key,
    "measurement_packages": {p: version(p) for p in ("jiwer", "accelerate")},
})

{
  "tesseract": {
    "clean_novel": {
      "key": "SF-177277",
      "family": "novel-00",
      "truth": "Balance payable 1,122.29",
      "prediction": "Balance payable 1,122.29",
      "raw_truth": "Balance payable 1,122.29",
      "raw_prediction": "Balance payable 1,122.29",
      "complete": null,
      "char_edits": 0,
      "chars": 24,
      "word_edits": 0,
      "words": 3,
      "exact": true
    },
    "degraded_novel": {
      "key": "SF-177277",
      "family": "novel-00",
      "truth": "Balance payable 1,122.29",
      "prediction": "Balance payable 1,122.29",
      "raw_truth": "Balance payable 1,122.29",
      "raw_prediction": "Balance payable 1,122.29",
      "complete": null,
      "char_edits": 0,
      "chars": 24,
      "word_edits": 0,
      "words": 3,
      "exact": true
    }
  },
  "frozen": {
    "clean_novel": {
      "key": "SF-177277",
      "family": "novel-00",
      "truth": "Balance payable 1,122.29",
      "prediction": "(13,13),(987,987)",
   